[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S05_numpy_indexing_filtrado.ipynb)

# Sesión 05 · Indexing y filtrado en 1D

**Módulo 2: NumPy** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Seleccionar elementos de un array con índices positivos y negativos, listas de índices y slicing con paso.
2. Filtrar un array con una máscara booleana y combinar condiciones con `&`, `|` y `~`.
3. Contar y resumir condiciones con `np.sum`, `np.mean`, `np.any` y `np.all`, y elegir valores con `np.where`.
4. Anticipar casos borde: filtros que no devuelven nada, `%` con negativos y el tipo del resultado.

## 📋 Qué debes saber antes
Sesión 4: crear arrays, operar con escalares y comparar elemento a elemento.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.
- Hoy tampoco hacen falta bucles, salvo dentro de las funciones del ejercicio 4 si los prefieres.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos de práctica y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión y las funciones que revisan tus respuestas.
import copy
import hashlib
import math

import numpy as np

rng = np.random.default_rng(42)

# ---------- Datos de práctica: ventas de tiendas ----------
meta_diaria = 3000
dias_mes = np.arange(1, 31)                     # el día 1 es lunes
ventas_mes = rng.integers(800, 5000, size=30)
ventas_mes[14] = 0                              # el día 15 la tienda cerró

productos = np.array(["polo básico", "polo estampado", "polo piqué", "jean clásico", "jean slim", "jean mom",
                      "casaca denim", "casaca polar", "casaca impermeable", "polo manga larga", "jean recto",
                      "casaca bomber"])
categorias = np.array([p.split()[0] for p in productos])
precios = np.round(rng.uniform(25, 220, size=12), 2)
stock = rng.integers(1, 40, size=12)
stock[[2, 7]] = 0
desfases = np.array([-9, -7, -3, -1, 0, 2, 6, 13])

# ---------- Datos de práctica: movimientos bancarios ----------
_n = 40
tipos = rng.choice(["deposito", "retiro", "pago"], size=_n, p=[0.3, 0.4, 0.3])
tipos[[6, 19, 31]] = "comision"
canales = np.where(tipos == "retiro", rng.choice(["cajero", "agencia"], size=_n), rng.choice(["app", "agencia"], size=_n))
montos = np.select(
    [tipos == "deposito", tipos == "retiro", tipos == "pago"],
    [rng.uniform(100, 3000, _n), -rng.uniform(20, 900, _n), -rng.uniform(10, 600, _n)],
    default=-rng.uniform(1, 15, _n),
).round(2)
montos[31] = -12.5                              # una comisión alta

_NOMBRES = ["meta_diaria", "dias_mes", "ventas_mes", "productos", "categorias", "precios", "stock",
            "desfases", "tipos", "canales", "montos"]
_D = copy.deepcopy({k: globals()[k] for k in _NOMBRES})
# Versiones en listas de Python: los verificadores recalculan con bucles, sin máscaras de NumPy.
_L = {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in _D.items()}

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _py(x):
    return x.item() if isinstance(x, np.generic) else x


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    v = _L["ventas_mes"]
    _esc(r, "primer_dia", v[0], "revisa el índice del primer elemento")
    _esc(r, "ultimo_dia", v[len(v) - 1], "revisa el índice del último elemento")
    _arr(r, "primera_semana", [v[i] for i in range(7)], "deberían ser los 7 primeros días")
    _arr(r, "ultima_semana", [v[i] for i in range(23, 30)], "deberían ser los 7 últimos días")
    _arr(r, "ventas_dias_pares", [x for d, x in zip(range(1, 31), v) if d % 2 == 0],
         "deberían ser las ventas de los días 2, 4, 6... (ojo: el día 2 está en la posición 1)")
    _arr(r, "invertido", list(reversed(v)), "deberían ser las ventas del último día al primero")
    _arr(r, "seleccion", [v[0], v[14], v[29]], "deberían ser los días 1, 15 y 30, en ese orden")
    _sin_cambios(r, "ventas_mes")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_slice_fuera": "25af31a57c2047d854d189042b0ecfb66843c4c19d3dd9ce1403e0be3826a240",
        "pred_indice_fuera": "bd1dddfaf233665e87fb493c0364dc67523ff4525fc194616e411b16fa707f7a",
        "pred_vista": "c31ede78e498dacb77b802b4ecc8b66f6b97246b0e4b9666f51c70c887559680",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2")
    v, meta = _L["ventas_mes"], _D["meta_diaria"]
    prod, cat, pre, st = _L["productos"], _L["categorias"], _L["precios"], _L["stock"]
    _arr(r, "mascara_alta", [x >= meta for x in v], "cada elemento debe responder si la venta alcanza la meta", tipos="b")
    _arr(r, "ventas_altas", [x for x in v if x >= meta], "deberían ser solo las ventas que alcanzan la meta")
    _arr(r, "dias_altos", [d for d, x in zip(range(1, 31), v) if x >= meta],
         "deberían ser los números de día (de `dias_mes`) que alcanzan la meta")
    _arr(r, "precios_jean", [p for c, p in zip(cat, pre) if c == "jean"], "deberían ser los precios de la categoría jean")
    _arr(r, "sin_stock", [n for n, s in zip(prod, st) if s == 0], "deberían ser los nombres con stock 0")
    _arr(r, "caros_con_stock", [n for n, p, s in zip(prod, pre, st) if p > 100 and s > 0],
         "deberían cumplirse las dos condiciones a la vez")
    _arr(r, "jean_o_casaca", [n for n, c in zip(prod, cat) if c in ("jean", "casaca")],
         "basta con que se cumpla una de las dos condiciones")
    _arr(r, "no_polo", [n for n, c in zip(prod, cat) if c != "polo"], "deberían ser todos los que no son polo")
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3")
    v, meta = _L["ventas_mes"], _D["meta_diaria"]
    altos = len([x for x in v if x >= meta])
    _esc(r, "n_dias_altos", altos, "cuenta los días que alcanzan la meta")
    _esc(r, "pct_dias_altos", round(altos * 100 / len(v), 1), "porcentaje de días que alcanzan la meta, con 1 decimal", tol=0.051)
    for nombre, ref in (("hubo_cierre", 0 in v), ("todos_con_venta", min(v) > 0)):
        x = r.var(nombre)
        if x is not _FALTA:
            if not isinstance(x, (bool, np.bool_)):
                r.mal(f"`{nombre}` debería ser `True` o `False` y es {type(x).__name__}.")
            elif bool(x) == ref:
                r.ok(f"`{nombre}` es correcto.")
            else:
                r.mal(f"`{nombre}` no es correcto; revisa si usaste `np.any` o `np.all` y la condición.")
    _arr(r, "etiquetas", ["alta" if x >= meta else "baja" for x in v], "revisa la condición y el orden de los dos valores", tipos="U")
    _arr(r, "posiciones_cierre", [i for i, x in enumerate(v) if x == 0],
         "deberían ser las posiciones (no los días) con venta 0; recuerda que `np.where` con un argumento devuelve una tupla")
    _arr(r, "bono", [round(x * 5 / 100, 2) if x >= meta else 0 for x in v],
         "5 % de la venta en los días que alcanzan la meta y 0 en los demás", tol=0.0051)
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    f = r.funcion("max_o_none")
    igual_py = lambda a, b: _igual(_py(a), b, 1e-9)
    if f is not _FALTA:
        casos = [("np.array([3, 9, 2])", np.array([3, 9, 2]), 9), ("np.array([])", np.array([]), None),
                 ("np.array([-5, -2])", np.array([-5, -2]), -2),
                 ("ventas_mes[ventas_mes > 10**6]", np.array(_D["ventas_mes"][_D["ventas_mes"] > 10**6]), None)]
        for texto, arr, esperado in casos:
            r.caso(f"max_o_none({texto})", f, (arr,), esperado=esperado, igual=igual_py,
                   motivo="con un array vacío debía devolver None" if esperado is None else "no es lo esperado")
    g = r.funcion("contar_entre")
    if g is not _FALTA:
        v = _D["ventas_mes"]
        casos = [("ventas_mes, 2000, 3000", (v, 2000, 3000)), ("np.array([]), 0, 10", (np.array([]), 0, 10)),
                 ("ventas_mes, 6000, 9000", (v, 6000, 9000)), ("np.array([-5, 0, 5]), -5, 0", (np.array([-5, 0, 5]), -5, 0))]
        for texto, args in casos:
            esperado = len([x for x in args[0].tolist() if args[1] <= x <= args[2]])
            r.caso(f"contar_entre({texto})", g, args, esperado=esperado, igual=igual_py,
                   motivo="sin coincidencias debía devolver 0" if esperado == 0 else "no es lo esperado (los dos límites se incluyen)")
    _arr(r, "dia_semana", [x - 7 * math.floor(x / 7) for x in _L["desfases"]],
         "cada valor debería quedar entre 0 y 6, también los negativos")
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_mod_neg": "7a20ad6fbd72c40ec9ce42fcce75b2afd01e02ef19725a1956c8b5a8b66eeefc",
        "pred_impar": "504fd320c035cf419fcfb0f0cbe7f3b478d10f78fda40022eeb35a86b23bce08",
        "pred_dtype_mascara": "057d91799f7a5eae5a47b35e84fef8977982f38f68a95047dcf51b47c1007132",
        "pred_dtype_filtro": "2f8dd469968d20ee46ee0fc632f0166ae995b6d83aa1ecfa34632232c2040b6f",
    })
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    t, c, m = _L["tipos"], _L["canales"], _L["montos"]
    filas = list(zip(t, c, m))
    _esc(r, "total_retiros", round(math.fsum(x for tt, _, x in filas if tt == "retiro"), 2),
         "suma los montos de tipo retiro (será negativo) y redondea a 2 decimales", tol=0.0051)
    _esc(r, "n_retiros_grandes_cajero", len([1 for tt, cc, x in filas if tt == "retiro" and cc == "cajero" and x < -300]),
         "cuenta los que cumplen las tres condiciones a la vez")
    _arr(r, "depositos_cajero", [x for tt, cc, x in filas if tt == "deposito" and cc == "cajero"],
         "deberían ser los montos de depósitos hechos en cajero")
    _esc(r, "total_depositos_cajero", 0.0 + math.fsum(x for tt, cc, x in filas if tt == "deposito" and cc == "cajero"),
         "suma `depositos_cajero`")
    _esc(r, "pct_app", round(len([1 for cc in c if cc == "app"]) * 100 / len(c), 1),
         "porcentaje de operaciones hechas por app, con 1 decimal", tol=0.051)
    _arr(r, "alerta", ["revisar" if x < -500 else "ok" for x in m], "revisa la condición y el orden de los dos textos", tipos="U")
    x = r.var("hay_comision_alta")
    if x is not _FALTA:
        ref = any(tt == "comision" and mm < -10 for tt, mm in zip(t, m))
        if isinstance(x, (bool, np.bool_)) and bool(x) == ref:
            r.ok("`hay_comision_alta` es correcto.")
        else:
            r.mal("`hay_comision_alta` no es correcto; debería ser `True` o `False` según si existe al menos una.")
    _esc(r, "pos_primer_retiro_grande", next((i for i, (tt, _, mm) in enumerate(filas) if tt == "retiro" and mm < -300), -1),
         "debería ser la posición del primer retiro menor que -300, o -1 si no hay")
    _sin_cambios(r, "tipos", "canales", "montos")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    v = _L["ventas_mes"]
    finde = [d % 7 in (6, 0) for d in range(1, 31)]
    _arr(r, "es_finde", finde, "sábados y domingos, sabiendo que el día 1 es lunes", tipos="b")
    _esc(r, "prom_finde", round(math.fsum(x for x, f in zip(v, finde) if f) / sum(finde), 2), "promedio de los días de fin de semana", tol=0.0051)
    _esc(r, "prom_habil", round(math.fsum(x for x, f in zip(v, finde) if not f) / (30 - sum(finde)), 2), "promedio de los días hábiles", tol=0.0051)
    _arr(r, "lunes", [v[i] for i in range(0, 30, 7)], "deberían ser las ventas de los días 1, 8, 15, 22 y 29")
    r.fin()


print("✅ Setup listo. Datos generados y verificadores cargados.")

### 📦 Tus datos de hoy
Los valores se generan con una semilla fija, así que siempre salen iguales. El mes empieza un lunes y el día 15 la tienda estuvo cerrada.

In [ ]:
print("🏪 Ventas de tiendas")
print("meta_diaria =", meta_diaria)
print("dias_mes    =", dias_mes)
print("ventas_mes  =", ventas_mes)
print()
for p, c, pr, s in zip(productos, categorias, precios, stock):
    print(f"   {p:<20} {c:<7} S/ {pr:>7.2f}  stock {s}")
print("desfases    =", desfases)
print()
print("🏦 Movimientos bancarios (tipo, canal, monto)")
for t, c, m in zip(tipos, canales, montos):
    print(f"   {t:<9} {c:<8} {m:>9.2f}")

---
## 1. Índices, listas de índices y slicing

### 📘 Concepto
Un array 1D se indexa como una lista:
- `a[0]` es el primero y `a[-1]` el último.
- `a[inicio:fin:paso]`: desde `inicio` hasta antes de `fin`, saltando de `paso` en `paso`. `a[::2]` toma uno sí y uno no; `a[::-1]` lo invierte.
- **Lista de índices** (*fancy indexing*): `a[[0, 3, 5]]` devuelve un array nuevo con esas posiciones, en el orden que pidas.

Dos detalles que no pasan con las listas:
- Un índice fuera de rango da `IndexError`, pero un **slice** fuera de rango devuelve un array vacío, sin error.
- Un slice es una **vista**: comparte datos con el original, así que modificar el slice modifica el original. Si necesitas independencia, usa `.copy()`.

In [ ]:
visitas_ej = np.array([120, 340, 90, 410, 205, 330, 180])
print(visitas_ej[0], visitas_ej[-1])
print(visitas_ej[2:5], visitas_ej[::2], visitas_ej[::-1])
print(visitas_ej[[6, 0, 3]])       # en el orden pedido
print(visitas_ej[10:])             # slice fuera de rango: vacío, sin error

copia_ej = visitas_ej[:3].copy()   # independiente del original
copia_ej[0] = 0
print(visitas_ej[0])               # el original no cambió

### ✍️ Tu turno · Ejercicio 1: elegir días
**Parte A.** Con `ventas_mes` (día 1 en la posición 0):
1. `primer_dia` y `ultimo_dia` (para el último usa un índice negativo).
2. `primera_semana` y `ultima_semana`: los 7 primeros y los 7 últimos días.
3. `ventas_dias_pares`: las ventas de los días 2, 4, 6, ..., 30, con slicing y paso.
4. `invertido`: las ventas del último día al primero.
5. `seleccion`: las ventas de los días 1, 15 y 30, con una lista de índices.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_slice_fuera` | `len(ventas_mes[40:])` | número o `"error"` |
| `pred_indice_fuera` | `ventas_mes[40]` | número o `"error"` |
| `pred_vista` | tras `a = np.array([1, 2, 3, 4])`, `b = a[:2]` y `b[0] = 99`, ¿cuánto vale `a[0]`? | número |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

El día `d` está en la posición `d - 1`. Escribe a mano las posiciones de los días 2, 4 y 6 para descubrir dónde empieza el slice.
</details>

<details><summary>💡 Pista 2</summary>

`ventas_dias_pares` empieza en la posición 1 y avanza de 2 en 2. Para `seleccion` usa doble corchete: el de indexar y el de la lista.
</details>

---
## 2. Filtrar con máscaras booleanas

### 📘 Concepto
Una **máscara** es un array de `bool` del mismo tamaño que los datos. `a[mascara]` devuelve solo los elementos donde la máscara es `True`.

La máscara puede salir de **otro** array del mismo tamaño: `nombres[precios > 100]` da los nombres de los productos caros.

Para combinar condiciones se usan operadores elemento a elemento (no `and`/`or`/`not`, que dan error con arrays):

| Operador | Significado |
|---|---|
| `&` | y |
| `\|` | o |
| `~` | no |

⚠️ **Cada condición va entre paréntesis**: `(a > 5) & (a < 10)`. Sin ellos, Python evalúa primero `&` y el resultado es otro o un error.

In [ ]:
nombres_ej = np.array(["polo", "jean", "gorra", "casaca", "medias"])
precios_ej = np.array([39.9, 129.0, 25.0, 189.5, 12.0])
stock_ej = np.array([10, 0, 5, 3, 0])

caros_ej = precios_ej > 100
print(caros_ej)
print(precios_ej[caros_ej])
print(nombres_ej[caros_ej])                          # la máscara sale de otro array
print(nombres_ej[(precios_ej > 20) & (stock_ej > 0)])
print(nombres_ej[(stock_ej == 0) | (precios_ej < 20)])
print(nombres_ej[~(stock_ej == 0)])

### ✍️ Tu turno · Ejercicio 2: filtros sobre ventas y productos
Con `ventas_mes`, `dias_mes` y `meta_diaria`:
1. `mascara_alta`: `True` en los días que alcanzan la meta.
2. `ventas_altas`: solo las ventas de esos días.
3. `dias_altos`: los **números de día** (de `dias_mes`) que alcanzan la meta.

Con `productos`, `categorias`, `precios` y `stock`:

4. `precios_jean`: los precios de la categoría `"jean"`.
5. `sin_stock`: los nombres de los productos con stock 0.
6. `caros_con_stock`: los nombres de los productos que cuestan más de 100 **y** tienen stock.
7. `jean_o_casaca`: los nombres de los productos de categoría jean **o** casaca.
8. `no_polo`: los nombres de los que **no** son de categoría polo, usando `~`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Todos los arrays de productos tienen el mismo tamaño y orden: una condición sobre `categorias` o `precios` sirve para filtrar `productos`.
</details>

<details><summary>💡 Pista 2</summary>

Revisa los paréntesis: `productos[(precios > 100) & (stock > 0)]`. Con `~`, la condición entera va entre paréntesis.
</details>

---
## 3. Contar, preguntar y elegir: `sum`, `any`, `all`, `where`

### 📘 Concepto
- `np.sum(mascara)` cuenta los `True` y `np.mean(mascara)` da la **proporción** (multiplícala por 100 para el porcentaje).
- `np.any(mascara)`: ¿al menos uno es `True`? `np.all(mascara)`: ¿todos lo son?
- `np.where(condicion, si, no)` arma un array nuevo eligiendo, elemento a elemento, el valor `si` donde la condición se cumple y `no` donde no.
- `np.where(condicion)`, con un solo argumento, devuelve **las posiciones** donde se cumple, dentro de una tupla: toma el primer elemento con `[0]`.

In [ ]:
stock_ej = np.array([10, 0, 5, 3, 0])
print(np.sum(stock_ej == 0), np.mean(stock_ej == 0))
print(np.any(stock_ej == 0), np.all(stock_ej >= 0))
print(np.where(stock_ej == 0, "reponer", "ok"))
print(np.where(stock_ej > 4, stock_ej * 2, 0))
print(np.where(stock_ej == 0))        # tupla con las posiciones
print(np.where(stock_ej == 0)[0])

### ✍️ Tu turno · Ejercicio 3: resumir el mes
Con `ventas_mes` y `meta_diaria`:
1. `n_dias_altos`: cuántos días alcanzaron la meta.
2. `pct_dias_altos`: qué porcentaje de días la alcanzó, redondeado a 1 decimal.
3. `hubo_cierre`: ¿hubo algún día con venta 0?
4. `todos_con_venta`: ¿todos los días tuvieron venta mayor que 0?
5. `etiquetas`: `"alta"` en los días que alcanzan la meta y `"baja"` en los demás.
6. `posiciones_cierre`: las posiciones de los días con venta 0.
7. `bono`: el 5 % de la venta en los días que alcanzan la meta y 0 en los demás, redondeado a 2 decimales.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Casi todo parte de la misma máscara: `ventas_mes >= meta_diaria`. `hubo_cierre` y `todos_con_venta` usan otra condición.
</details>

<details><summary>💡 Pista 2</summary>

Para `bono`, el segundo argumento de `np.where` puede ser un array: `ventas_mes * 0.05`. Redondea el resultado completo con `np.round`.
</details>

---
## 4. Casos borde

### 📘 Concepto
- **Filtros vacíos.** Si ningún elemento cumple, `a[mascara]` devuelve un array vacío (`size == 0`) sin error. Pero sobre un array vacío `np.max` da error, `np.mean` da `nan` con un aviso y `np.sum` da 0. Revisa `a.size` antes de resumir.
- **`%` con negativos.** Igual que en Python, el resultado tiene el signo del divisor: `-3 % 7` es `4`. Es útil para "dar la vuelta": un desfase de -3 días desde el lunes (0) cae en viernes (4).
- **Tipo del resultado.** Una comparación siempre da `bool`. Filtrar no cambia el `dtype`, aunque el resultado esté vacío.

In [ ]:
precios_ej = np.array([39.9, 129.0, 25.0])
vacio_ej = precios_ej[precios_ej > 1000]
print(vacio_ej, vacio_ej.size, vacio_ej.dtype, np.sum(vacio_ej))
if vacio_ej.size == 0:
    print("No hay productos que cumplan")

print(np.array([-8, -1, 0, 5, 9]) % 7)

### ✍️ Tu turno · Ejercicio 4: funciones a prueba de vacíos
**Parte A.**
1. `max_o_none(arr)`: el máximo de `arr` o `None` si está vacío.
2. `contar_entre(arr, bajo, alto)`: cuántos elementos están entre `bajo` y `alto`, **ambos incluidos**. Sin coincidencias o con un array vacío, devuelve 0.
3. `dia_semana`: para cada valor de `desfases` (días de diferencia respecto de un lunes), el día de la semana del 0 (lunes) al 6 (domingo), usando `%`.

El verificador probará las funciones con arrays vacíos, negativos y búsquedas sin resultado.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_mod_neg` | el valor de `np.array([-3]) % 7` | número |
| `pred_impar` | ¿`np.array([-3]) % 2 == 1` da `True`? | `True` o `False` |
| `pred_dtype_mascara` | `str((np.arange(3) > 1).dtype)` | texto |
| `pred_dtype_filtro` | el `dtype` de `precios[precios > 1000]` (como texto) | texto |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

En `max_o_none`, revisa `arr.size` antes de llamar a `max`. En `contar_entre`, combina dos condiciones con `&` y cuenta con `np.sum`.
</details>

<details><summary>💡 Pista 2</summary>

`dia_semana` es una sola operación sobre `desfases`. Comprueba a mano un caso: -3 días desde un lunes es viernes.
</details>

---
## 🏋️ Reto final: filtros sobre movimientos bancarios
`tipos`, `canales` y `montos` describen 40 movimientos (misma posición, mismo movimiento). Los depósitos son positivos; retiros, pagos y comisiones, negativos. Resuelve **sin bucles**:

1. `total_retiros`: suma de los montos de tipo `"retiro"`, redondeada a 2 decimales.
2. `n_retiros_grandes_cajero`: cuántos retiros se hicieron en `"cajero"` por más de 300 soles (monto menor que -300).
3. `depositos_cajero`: los montos de depósitos hechos en cajero, y `total_depositos_cajero`: su suma. (¿Qué pasa si no hay ninguno?)
4. `pct_app`: porcentaje de operaciones hechas por `"app"`, con 1 decimal.
5. `alerta`: `"revisar"` en los movimientos menores que -500 y `"ok"` en los demás.
6. `hay_comision_alta`: ¿hay alguna comisión mayor que 10 soles (monto menor que -10)?
7. `pos_primer_retiro_grande`: la posición del primer retiro menor que -300, o `-1` si no hay ninguno.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Arma primero las máscaras que vas a reutilizar, por ejemplo `es_retiro = tipos == "retiro"`, y combínalas con `&`.
</details>

<details><summary>💡 Pista 2</summary>

Para el punto 7: `np.where(mascara)[0]` da todas las posiciones; si su `size` es 0 no hay ninguna, y si no, la primera es `[0]`.
</details>

---
## 🚀 Nivel pro (opcional)
El mes empieza un lunes.
1. `es_finde`: máscara con `True` en sábados y domingos, calculada con `dias_mes` y `%` (sin escribir los días a mano).
2. `prom_finde` y `prom_habil`: la venta promedio de los fines de semana y de los días hábiles, con 2 decimales.
3. `lunes`: las ventas de todos los lunes del mes, usando slicing con paso.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Usar índices negativos, slicing con paso y listas de índices.
- [ ] Explicar por qué un slice es una vista y cuándo hace falta `.copy()`.
- [ ] Filtrar un array con una máscara que sale de otro array del mismo tamaño.
- [ ] Combinar condiciones con `&`, `|` y `~`, y explicar por qué van entre paréntesis.
- [ ] Contar y sacar proporciones con `np.sum` y `np.mean` sobre una máscara.
- [ ] Explicar la diferencia entre `np.where` con tres argumentos y con uno.
- [ ] Anticipar qué pasa al resumir un filtro vacío.
- [ ] Explicar qué hace `%` con números negativos.

**Próxima sesión (S06):** arrays 2D, `reshape` y el parámetro `axis`.